# Notebook 7 – Batch Normalization and Dropout



In [1]:
import time
import numpy as np
import torch
from torch import nn
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

PyTorch: 2.14.0+cpu | device: cpu


## Concept and modes
Input standardization transforms images before the model. BatchNorm normalizes intermediate activations using batch statistics during training and running statistics during evaluation, with learned scale/shift. Dropout randomly zeros activations during training and is disabled at evaluation. Call `model.train()` before optimization and `model.eval()` for validation/inference. `eval()` changes module behavior but does not itself disable gradients; use `torch.no_grad()` too.

In [2]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader,TensorDataset
import torch.nn.functional as F
d=load_digits(); x=torch.tensor(d.images,dtype=torch.float32).unsqueeze(1)/16; x=F.interpolate(x,(16,16),mode='bilinear',align_corners=False); y=torch.tensor(d.target,dtype=torch.long)
a,b,ya,yb=train_test_split(x,y,test_size=.2,random_state=42,stratify=y); tr=DataLoader(TensorDataset(a,ya),128,shuffle=True); va=DataLoader(TensorDataset(b,yb),256)
class RegCNN(nn.Module):
 def __init__(self,bn=False,drop=0.):
  super().__init__(); layers=[]; c=1
  for out in (16,32):
   layers.append(nn.Conv2d(c,out,3,padding=1));
   if bn: layers.append(nn.BatchNorm2d(out))
   layers += [nn.ReLU(),nn.MaxPool2d(2)]; c=out
  self.features=nn.Sequential(*layers,nn.AdaptiveAvgPool2d((2,2))); self.head=nn.Sequential(nn.Flatten(),nn.Dropout(drop),nn.Linear(128,10))
 def forward(self,x): return self.head(self.features(x))
for label,bn,drop in [('Baseline',False,0.),('BatchNorm',True,0.),('Dropout',False,.4),('BN + Dropout',True,.4)]:
 torch.manual_seed(42)
 m=RegCNN(bn,drop).to(device); opt=torch.optim.Adam(m.parameters(),lr=1e-3); start=time.perf_counter()
 for _ in range(2):
  m.train()
  for xb,y in tr:
   xb,y=xb.to(device),y.to(device); opt.zero_grad(); loss=nn.functional.cross_entropy(m(xb),y); loss.backward(); opt.step()
 m.eval(); good=total=0
 with torch.no_grad():
  for xb,y in va: good+=(m(xb.to(device)).argmax(1).cpu()==y).sum().item(); total+=len(y)
 print(label,'val_acc',round(good/total,3),'seconds',round(time.perf_counter()-start,2))

Baseline val_acc 0.294 seconds 0.63
BatchNorm val_acc 0.358 seconds 0.66
Dropout val_acc 0.247 seconds 0.99
BN + Dropout val_acc 0.344 seconds 0.94


## Interpret the result
Compare train and validation curves, not just one score. BatchNorm often helps optimization, but tiny batches may make estimates noisy. Dropout may reduce overfitting but too much can underfit. Repeat seeds.

